# Guardar Modelo de Producción — Fase B

Reentrena el modelo final con las 151 features y lo guarda en `models/`
reemplazando el modelo de Fase A para que la API use el mejor modelo.

In [1]:
import pandas as pd
import numpy as np
import joblib
import json
import warnings
warnings.filterwarnings('ignore')

import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, recall_score

RANDOM_STATE  = 42
DATA_PATH     = '../data/raw/'
PROCESSED_PATH = '../data/processed/'
MODEL_PATH    = '../models/'

print('Librerías cargadas.')

Librerías cargadas.


## 1. Armar el dataset completo (mismo proceso que notebook 11)

In [2]:
df = pd.read_csv(DATA_PATH + 'application_train.csv')

missing_pct = df.isnull().mean()
cols_to_drop = missing_pct[missing_pct > 0.4].index.tolist()
df = df.drop(columns=cols_to_drop)

df['AGE_YEARS'] = df['DAYS_BIRTH'].abs() / 365
df = df.drop(columns=['DAYS_BIRTH'])
df['FLAG_EMPLOYED_ANOMALY'] = (df['DAYS_EMPLOYED'] == 365243).astype(int)
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)

cat_cols = df.select_dtypes(include='object').columns.tolist()
for col in cat_cols:
    df[col] = df[col].astype('category')

bureau_feat = pd.read_parquet(PROCESSED_PATH + 'bureau_features.parquet')
prev_feat   = pd.read_parquet(PROCESSED_PATH + 'prev_app_features.parquet')
inst_feat   = pd.read_parquet(PROCESSED_PATH + 'installments_features.parquet')
pos_feat    = pd.read_parquet(PROCESSED_PATH + 'pos_cash_features.parquet')
cc_feat     = pd.read_parquet(PROCESSED_PATH + 'credit_card_features.parquet')

df = df.merge(bureau_feat, on='SK_ID_CURR', how='left')
df = df.merge(prev_feat,   on='SK_ID_CURR', how='left')
df = df.merge(inst_feat,   on='SK_ID_CURR', how='left')
df = df.merge(pos_feat,    on='SK_ID_CURR', how='left')
df = df.merge(cc_feat,     on='SK_ID_CURR', how='left')
df = df.drop(columns=['SK_ID_CURR'])

print(f'Dataset: {df.shape} — {df.shape[1]-1} features')

Dataset: (307511, 152) — 151 features


## 2. Entrenar sobre TODO el dataset (sin split)

El modelo de producción se entrena con el 100% de los datos disponibles.
Ya validamos las métricas en el notebook 11 — aquí aprovechamos todos los ejemplos.

In [3]:
X = df.drop(columns=['TARGET'])
y = df['TARGET']

feature_names = X.columns.tolist()
categorical_cols = X.select_dtypes(include='category').columns.tolist()

neg = (y == 0).sum()
pos = (y == 1).sum()
scale_pos_weight = neg / pos

# n_estimators = best_iteration del notebook 11 (234 árboles)
# No necesitamos early stopping — ya sabemos el número óptimo
model = lgb.LGBMClassifier(
    n_estimators=234,
    learning_rate=0.05,
    num_leaves=31,
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1
)

print('Entrenando sobre el 100% de los datos...')
model.fit(X, y)

# Verificar con train score (referencia, no métrica real)
y_proba_train = model.predict_proba(X)[:, 1]
train_auc = roc_auc_score(y, y_proba_train)
print(f'Train AUC (referencia): {train_auc:.4f}')
print(f'Árboles: {model.n_estimators_}')
print('Modelo listo.')

Entrenando sobre el 100% de los datos...
Train AUC (referencia): 0.8273
Árboles: 234
Modelo listo.


## 3. Guardar modelo y metadatos

In [4]:
joblib.dump(model, MODEL_PATH + 'lgbm_credit_risk.pkl')
print(f'Modelo guardado en models/lgbm_credit_risk.pkl')

import os
size_mb = os.path.getsize(MODEL_PATH + 'lgbm_credit_risk.pkl') / (1024 * 1024)
print(f'Tamaño: {size_mb:.2f} MB')

metadata = {
    'model': 'LGBMClassifier',
    'fase': 'B',
    'version': '2.0.0',
    'n_estimators': 234,
    'learning_rate': 0.05,
    'num_leaves': 31,
    'scale_pos_weight': round(scale_pos_weight, 4),
    'metrics': {
        'roc_auc_validation': 0.7765,
        'recall_validation': 0.6882,
        'fase_a_roc_auc': 0.7561,
        'baseline_roc_auc': 0.7435
    },
    'features': {
        'total': len(feature_names),
        'from_application': 72,
        'from_secondary_tables': 79,
        'tables': {
            'bureau': 17,
            'previous_application': 16,
            'installments_payments': 15,
            'pos_cash_balance': 14,
            'credit_card_balance': 17
        }
    },
    'preprocessing': {
        'categorical_cols': categorical_cols,
        'missing_threshold_dropped': 0.4,
        'days_employed_anomaly': 365243
    },
    'feature_names': feature_names
}

with open(MODEL_PATH + 'model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('Metadatos guardados en models/model_metadata.json')
print(f'\nResumen:')
print(f'  Features: {len(feature_names)}')
print(f'  Categóricas: {len(categorical_cols)}')
print(f'  ROC-AUC validación: 0.7765')
print(f'  Recall validación:  0.6882')

Modelo guardado en models/lgbm_credit_risk.pkl
Tamaño: 0.81 MB
Metadatos guardados en models/model_metadata.json

Resumen:
  Features: 151
  Categóricas: 12
  ROC-AUC validación: 0.7765
  Recall validación:  0.6882
